# Construção de Oráculos Quânticos

## Objetivo
Aprender a construir oráculos customizados para algoritmos quânticos.

## O que é um Oráculo?
Um oráculo é uma "caixa preta" que codifica informação sobre o problema. Em computação quântica, oráculos são implementados como operadores unitários.

## Tipos de Oráculos
- **Phase Oracle**: Inverte a fase do estado alvo (usado em Grover)
- **Bit-flip Oracle**: Inverte um qubit auxiliar (usado em Deutsch-Jozsa)

## Referências
- Livro: Capítulo 9
- [Qiskit Textbook - Oracles](https://qiskit.org/textbook/ch-algorithms/grover.html#2.-The-Oracle-)

In [ ]:
# Imports
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np

## 1. Phase Oracle (Oráculo de Fase)

Inverte a fase do estado alvo: |x⟩ → -|x⟩ se x é o alvo.

In [ ]:
def create_phase_oracle(n_qubits: int, target: str) -> QuantumCircuit:
    """
    Cria um oráculo de fase que marca o estado alvo com fase -1.
    
    Args:
        n_qubits: número de qubits
        target: estado alvo em binário (ex: "101")
    
    Returns:
        QuantumCircuit: oráculo de fase
    """
    oracle = QuantumCircuit(n_qubits, name=f"Oracle_{target}")
    
    # Converter para little-endian (Qiskit usa esta convenção)
    target_bits = target[::-1]
    
    # Aplicar X nos qubits que devem ser |0⟩ no target
    for i, bit in enumerate(target_bits):
        if bit == '0':
            oracle.x(i)
    
    # Aplicar MCZ (Multi-Controlled Z)
    # Decomposição: H no último qubit, MCX, H no último qubit
    oracle.h(n_qubits - 1)
    oracle.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    oracle.h(n_qubits - 1)
    
    # Desfazer os X
    for i, bit in enumerate(target_bits):
        if bit == '0':
            oracle.x(i)
    
    return oracle

# Testar para |11⟩
oracle_11 = create_phase_oracle(2, "11")
print("Oráculo de fase para |11⟩:")
print(oracle_11.draw())

In [ ]:
# Verificar que o oráculo inverte a fase corretamente
def test_phase_oracle(oracle: QuantumCircuit, target: str):
    """Verifica se o oráculo inverte apenas a fase do estado alvo."""
    n = oracle.num_qubits
    
    # Criar superposição uniforme
    qc = QuantumCircuit(n)
    qc.h(range(n))
    
    # Estado antes do oráculo
    state_before = Statevector.from_instruction(qc)
    print(f"Antes do oráculo:")
    for i, amp in enumerate(state_before.data):
        if abs(amp) > 0.01:
            print(f"  |{format(i, f'0{n}b')}⟩: {amp:.3f}")
    
    # Aplicar oráculo
    qc.compose(oracle, inplace=True)
    state_after = Statevector.from_instruction(qc)
    
    print(f"\nDepois do oráculo (alvo: |{target}⟩):")
    for i, amp in enumerate(state_after.data):
        if abs(amp) > 0.01:
            state_str = format(i, f'0{n}b')[::-1]  # Converter de little-endian
            marker = " ← MARCADO" if state_str == target else ""
            print(f"  |{state_str}⟩: {amp:.3f}{marker}")

test_phase_oracle(oracle_11, "11")

## 2. Bit-Flip Oracle (Oráculo de Inversão de Bit)

Inverte um qubit auxiliar quando a entrada satisfaz uma condição.

In [ ]:
def create_bitflip_oracle(n_qubits: int, target: str) -> QuantumCircuit:
    """
    Cria um oráculo bit-flip: |x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩
    onde f(x) = 1 se x == target, 0 caso contrário.
    
    Args:
        n_qubits: número de qubits de entrada (sem contar o auxiliar)
        target: estado alvo em binário
    
    Returns:
        QuantumCircuit: oráculo com n_qubits + 1 qubits
    """
    # n qubits de entrada + 1 qubit auxiliar
    oracle = QuantumCircuit(n_qubits + 1, name=f"BitFlip_{target}")
    
    target_bits = target[::-1]  # Little-endian
    
    # Aplicar X nos qubits que devem ser |0⟩
    for i, bit in enumerate(target_bits):
        if bit == '0':
            oracle.x(i)
    
    # MCX: inverte auxiliar se todos controles são |1⟩
    oracle.mcx(list(range(n_qubits)), n_qubits)
    
    # Desfazer os X
    for i, bit in enumerate(target_bits):
        if bit == '0':
            oracle.x(i)
    
    return oracle

# Testar para |10⟩
oracle_bf = create_bitflip_oracle(2, "10")
print("Oráculo bit-flip para |10⟩:")
print(oracle_bf.draw())

## 3. Oráculo para Múltiplos Alvos

Marca mais de um estado como alvo.

In [ ]:
def create_multi_target_oracle(n_qubits: int, targets: list) -> QuantumCircuit:
    """
    Cria um oráculo de fase que marca múltiplos estados.
    
    Args:
        n_qubits: número de qubits
        targets: lista de estados alvo em binário
    
    Returns:
        QuantumCircuit: oráculo composto
    """
    oracle = QuantumCircuit(n_qubits, name=f"MultiOracle")
    
    for target in targets:
        # Criar sub-oráculo para cada alvo
        sub_oracle = create_phase_oracle(n_qubits, target)
        oracle.compose(sub_oracle, inplace=True)
    
    return oracle

# Marcar |00⟩ e |11⟩
multi_oracle = create_multi_target_oracle(2, ["00", "11"])
print("Oráculo para |00⟩ e |11⟩:")
print(multi_oracle.decompose().draw())

In [ ]:
# Verificar oráculo multi-alvo
def verify_multi_oracle(oracle: QuantumCircuit, targets: list):
    """Verifica oráculo multi-alvo."""
    n = oracle.num_qubits
    
    qc = QuantumCircuit(n)
    qc.h(range(n))
    qc.compose(oracle, inplace=True)
    
    state = Statevector.from_instruction(qc)
    
    print(f"Alvos: {targets}")
    print("Estados após oráculo:")
    for i, amp in enumerate(state.data):
        if abs(amp) > 0.01:
            state_str = format(i, f'0{n}b')[::-1]
            marker = " ← MARCADO" if state_str in targets else ""
            sign = "+" if amp.real > 0 else "-"
            print(f"  {sign}|{state_str}⟩{marker}")

verify_multi_oracle(multi_oracle, ["00", "11"])

## 4. Oráculo com Condição Aritmética

Exemplo: marcar estados onde x < 3 (para 2 qubits: |00⟩, |01⟩, |10⟩)

In [ ]:
def create_less_than_oracle(n_qubits: int, threshold: int) -> QuantumCircuit:
    """
    Cria oráculo que marca estados onde o valor decimal < threshold.
    
    Args:
        n_qubits: número de qubits
        threshold: valor limite
    
    Returns:
        QuantumCircuit: oráculo
    """
    # Encontrar todos os estados que satisfazem a condição
    targets = []
    for i in range(2**n_qubits):
        if i < threshold:
            targets.append(format(i, f'0{n_qubits}b'))
    
    print(f"Estados com valor < {threshold}: {targets}")
    return create_multi_target_oracle(n_qubits, targets)

# Marcar x < 3 (para 2 qubits: 00, 01, 10)
less_than_oracle = create_less_than_oracle(2, 3)
verify_multi_oracle(less_than_oracle, ["00", "01", "10"])

## 5. Exercício: Oráculo de Paridade

Crie um oráculo que marca estados com número ímpar de 1s.

In [ ]:
def create_odd_parity_oracle(n_qubits: int) -> QuantumCircuit:
    """
    Cria oráculo que marca estados com paridade ímpar.
    
    Para 3 qubits, marca: |001⟩, |010⟩, |100⟩, |111⟩
    """
    targets = []
    for i in range(2**n_qubits):
        binary = format(i, f'0{n_qubits}b')
        if binary.count('1') % 2 == 1:  # Paridade ímpar
            targets.append(binary)
    
    print(f"Estados com paridade ímpar: {targets}")
    return create_multi_target_oracle(n_qubits, targets)

# Testar para 3 qubits
parity_oracle = create_odd_parity_oracle(3)
expected_targets = ["001", "010", "100", "111"]
verify_multi_oracle(parity_oracle, expected_targets)

## 6. Análise de Complexidade de Oráculos

Quantas portas são necessárias?

In [ ]:
def analyze_oracle_complexity(oracle: QuantumCircuit):
    """Analisa complexidade do oráculo."""
    decomposed = oracle.decompose()
    
    # Contar portas
    gate_counts = {}
    for instruction in decomposed.data:
        gate_name = instruction.operation.name
        gate_counts[gate_name] = gate_counts.get(gate_name, 0) + 1
    
    print(f"Análise do oráculo:")
    print(f"  Qubits: {oracle.num_qubits}")
    print(f"  Profundidade: {decomposed.depth()}")
    print(f"  Contagem de portas:")
    for gate, count in sorted(gate_counts.items()):
        print(f"    {gate}: {count}")
    print(f"  Total: {sum(gate_counts.values())} portas")

# Analisar oráculos criados
print("=" * 50)
print("Oráculo simples (1 alvo):")
analyze_oracle_complexity(create_phase_oracle(3, "101"))

print("\n" + "=" * 50)
print("Oráculo multi-alvo (4 alvos):")
analyze_oracle_complexity(create_odd_parity_oracle(3))

## 7. Conclusão

Responda:
- Qual a diferença entre phase oracle e bit-flip oracle?
- Por que a complexidade do oráculo é importante?
- Como oráculos são usados em algoritmos práticos?

**Resposta:**

1. **Phase vs Bit-flip Oracle:**
   - **Phase oracle**: Aplica fase -1 ao estado alvo, mantendo norma. Usado em Grover.
   - **Bit-flip oracle**: Inverte um qubit auxiliar. Usado em Deutsch-Jozsa.
   - Conversão: bit-flip com auxiliar em |-⟩ equivale a phase oracle.

2. **Importância da complexidade:**
   - Oráculos muito complexos podem anular o speedup quântico.
   - Em hardware real, portas multi-controladas (MCX) são decompostas em muitas portas básicas.
   - O "custo oculto" do oráculo deve ser considerado na análise de algoritmos.

3. **Aplicações práticas:**
   - **Busca**: Grover usa oráculo para marcar soluções.
   - **Otimização**: QAOA codifica função objetivo no oráculo.
   - **Criptografia**: Algoritmo de Simon usa oráculo de função.